# Text Vectorizer: Bag of Words, TF-IDF, dan Word Embedding (Word2Vec)

Notebook ini menjalankan ulang dan memverifikasi tiga teknik dasar text vectorizer yang dipakai untuk mengubah teks jadi representasi numerik yang bisa diproses model machine learning:

1. Bag of Words (BoW)
2. TF-IDF
3. Word Embedding (Word2Vec)


## 0. Instalasi & Import Library

In [ ]:
!pip install -q scikit-learn gensim Sastrawi joblib

In [2]:
import json
import re
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import euclidean_distances, cosine_similarity
from gensim.models import Word2Vec
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import joblib

np.set_printoptions(precision=4, suppress=True)
print("Semua library berhasil diimpor.")

Semua library berhasil diimpor.


## 1. Bag of Words (BoW)

BoW mengubah teks menjadi vektor berdasarkan **frekuensi kemunculan kata**, tanpa memperhatikan urutan kata.

In [3]:
documents = ["saya suka machine learning", "machine learning adalah masa depan"]

bow_vectorizer = CountVectorizer()
bow_vectors = bow_vectorizer.fit_transform(documents)

print("Vocabulary:", bow_vectorizer.get_feature_names_out())
print("Matriks BoW:")
print(bow_vectors.toarray())

Vocabulary: ['adalah' 'depan' 'learning' 'machine' 'masa' 'saya' 'suka']
Matriks BoW:
[[0 0 1 1 0 1 1]
 [1 1 1 1 1 0 0]]


**Cara baca:** setiap baris = satu dokumen, setiap kolom = satu kata di vocabulary (urut alfabetis). Angka menunjukkan berapa kali kata itu muncul di dokumen tersebut. Karena kedua kalimat contoh sama-sama pendek dan tidak ada kata berulang, semua nilai di sini 0 atau 1 — BoW baru terlihat bedanya kalau ada kata yang muncul berkali-kali dalam satu dokumen.

## 2. TF-IDF (Term Frequency–Inverse Document Frequency)

TF-IDF memberi bobot lebih rendah untuk kata yang terlalu umum (muncul di banyak dokumen) dan bobot lebih tinggi untuk kata yang unik.

In [4]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_vectors = tfidf_vectorizer.fit_transform(documents)

print("Vocabulary:", tfidf_vectorizer.get_feature_names_out())
print("Matriks TF-IDF:")
print(tfidf_vectors.toarray())

Vocabulary: ['adalah' 'depan' 'learning' 'machine' 'masa' 'saya' 'suka']
Matriks TF-IDF:
[[0.     0.     0.4099 0.4099 0.     0.5762 0.5762]
 [0.4992 0.4992 0.3552 0.3552 0.4992 0.     0.    ]]


**Perbandingan dengan BoW:** kata `"machine"` dan `"learning"` muncul di kedua dokumen, sehingga bobot TF-IDF-nya (≈0.35–0.41) lebih rendah dibanding kata yang cuma muncul di satu dokumen seperti `"saya"`, `"suka"` (≈0.58) atau `"adalah"`, `"masa"`, `"depan"` (≈0.50). Ini yang dimaksud "kata umum diberi bobot lebih rendah" — BoW tidak bisa membedakan ini karena BoW cuma menghitung, tidak mempertimbangkan seberapa umum kata itu di seluruh korpus.

## 3. Word Embedding (Word2Vec)

Word2Vec memetakan kata ke vektor berdimensi rendah berdasarkan konteks kemunculannya, menggunakan neural network kecil (CBOW/Skip-gram).

> ⚠️ **Catatan penting:** contoh di bawah ini sengaja memakai korpus yang sama seperti di atas (cuma 2 kalimat pendek) supaya konsisten dengan materi aslinya. Tapi ini juga alasan kenapa **hasilnya tidak bisa dipakai sebagai bukti bahwa Word2Vec "memahami makna"** — Word2Vec butuh korpus jauh lebih besar (biasanya jutaan kata) untuk menghasilkan embedding yang benar-benar merepresentasikan makna semantik. Dengan cuma 2 kalimat, angka vektor di bawah ini pada dasarnya masih acak/belum terlatih secara bermakna.

In [5]:
sentences = [
    ["saya", "suka", "machine", "learning"],
    ["machine", "learning", "adalah", "masa", "depan"],
]

w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=1, seed=42)

vec_machine = w2v_model.wv["machine"]
print("Dimensi vektor 'machine':", vec_machine.shape)
print("5 nilai pertama:", vec_machine[:5])
print()
print("Kata paling 'mirip' dengan 'machine' (menurut model kecil ini):")
print(w2v_model.wv.most_similar("machine", topn=3))

Dimensi vektor 'machine': (100,)
5 nilai pertama: [ 0.0067 -0.006   0.0061 -0.0098  0.0059]

Kata paling 'mirip' dengan 'machine' (menurut model kecil ini):
[('learning', 0.09104208648204803), ('saya', 0.02587900310754776), ('adalah', 0.005806456319987774)]


**Interpretasi jujur:** similarity score di atas semuanya sangat kecil (di bawah 0.1) — ini bukan hasil yang bisa diandalkan, cuma ilustrasi cara pakai API-nya. Untuk word embedding yang benar-benar berguna, materi selanjutnya di bootcamp biasanya memakai model pre-trained (mis. GloVe, FastText, atau embedding dari model Transformer) daripada melatih Word2Vec dari nol dengan data sedikit.

---

**Sumber materi:** rubythalib.ai AI Engineer Bootcamp — modul NLP (mentor: Muhammad Ikhwan Fathulloh).
**Catatan:** seluruh contoh kode di notebook ini telah dijalankan ulang dan diverifikasi secara independen; bukan disalin dari slide.

Proyek praktik penerapan TF-IDF (sistem tanya-jawab FAQ) dibahas terpisah di repo lain: `faq-search-engine-tfidf-sastrawi-arielshakaramiro`.
